In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict,Annotated
import os
import operator
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

True

In [2]:
endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )
model=ChatHuggingFace(llm=endpoint)

c:\Users\Kashish\Downloads\UPCON26_PHP (5)\Agentic-AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [4]:
def generate_joke(state:JokeState):
    prompt=f"""generate a joke on the topic {state['topic']}"""
    response=model.invoke(prompt).content
    return {"joke":response}

In [5]:
def generate_explaination(state:JokeState):
    prompy=f"""explain the joke {state['joke']}"""
    response=model.invoke(prompy).content
    return {"explaination":response}

In [6]:
graph=StateGraph(JokeState)
graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explaination",generate_explaination)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","generate_explaination")
graph.add_edge("generate_explaination",END)

checkpointer=MemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {'configurable': {'thread_id': '1'}}
initial_state={
    "topic":"Money"
}

workflow.invoke(initial_state,config=config1)


{'topic': 'Money',
 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)",
 'explaination': 'A classic play on words!\n\nThe joke relies on a double meaning of "flat":\n\n1. A dollar bill can be physically flat, meaning it\'s a two-dimensional piece of paper with a flat surface.\n2. The phrase "feeling a little flat" is an idiom that means feeling unwell or lacking energy, similar to feeling "under the weather".\n\nSo, the joke is saying that the dollar bill went to the doctor because it was feeling unwell (flat), but also making a pun on the fact that it\'s a flat piece of paper (flat). The humor comes from the unexpected twist on the usual meaning of "flat".'}

In [8]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Money', 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)", 'explaination': 'A classic play on words!\n\nThe joke relies on a double meaning of "flat":\n\n1. A dollar bill can be physically flat, meaning it\'s a two-dimensional piece of paper with a flat surface.\n2. The phrase "feeling a little flat" is an idiom that means feeling unwell or lacking energy, similar to feeling "under the weather".\n\nSo, the joke is saying that the dollar bill went to the doctor because it was feeling unwell (flat), but also making a pun on the fact that it\'s a flat piece of paper (flat). The humor comes from the unexpected twist on the usual meaning of "flat".'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b29-8ba3-6878-8002-07bd7daa7759'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-16T09:40:04.983205+00:00', parent_conf

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Money', 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)", 'explaination': 'A classic play on words!\n\nThe joke relies on a double meaning of "flat":\n\n1. A dollar bill can be physically flat, meaning it\'s a two-dimensional piece of paper with a flat surface.\n2. The phrase "feeling a little flat" is an idiom that means feeling unwell or lacking energy, similar to feeling "under the weather".\n\nSo, the joke is saying that the dollar bill went to the doctor because it was feeling unwell (flat), but also making a pun on the fact that it\'s a flat piece of paper (flat). The humor comes from the unexpected twist on the usual meaning of "flat".'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b29-8ba3-6878-8002-07bd7daa7759'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-16T09:40:04.983205+00:00', parent_con